In [1]:
import arcpy
import pandas as pd
import os
from collections import defaultdict, deque
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from arcpy.sa import *

## Extract static attributes from DEM

In [2]:
# Set the workspace (change the path to your workspace)
arcpy.env.workspace = r"C:\data_gnn_ver2"
arcpy.env.overwriteOutput = True

# Check out the Spatial Analyst extension
arcpy.CheckOutExtension("Spatial")

# Define the input DEM and shapefile
dem = "target_dem.tif"
shapefile = "sites22.shp"

# Calculate slope from DEM
slope_raster = Slope(dem, "DEGREE")
slope_raster_name = "slope.tif"
slope_raster.save(slope_raster_name)

# Calculate aspect from DEM
aspect_raster = Aspect(dem)
aspect_raster_name = "aspect.tif"
aspect_raster.save(aspect_raster_name)

# Extract DEM elevation, slope, and aspect values to points
arcpy.sa.ExtractMultiValuesToPoints(shapefile, [[dem, "Elevation"], [slope_raster_name, "Slope"], [aspect_raster_name, "Aspect"]], "NONE")

# Convert updated shapefile to DataFrame
# First, get a list of fields from the shapefile
field_names = [f.name for f in arcpy.ListFields(shapefile) if f.type != 'Geometry']
# Use a search cursor to iterate rows and create a list of lists from the attributes
data = [row for row in arcpy.da.SearchCursor(shapefile, field_names)]
# Create a DataFrame
df = pd.DataFrame(data, columns=field_names)


df

,FID,Station_Na,Station_En,Station_Co,Basin_Code,Start_Day_,Last_Day_o,latitude,longitude,NEAR_FID,NEAR_DIST,NEAR_X,NEAR_Y,NEAR_ANGLE,upDist,rid,ratio,locID,netID,pid,Elevation,Slope,Elevatio_1,Slope_1,Aspect,Elevatio_2,Slope_12,Aspect_1
0,0,평림댐,PYUNGRIMDAM,5002201,200214,200709,202402,35.303889,126.670833,19,0.002132,126.668792,35.303274,-163.242748,0.120817,18,0.734233,1,106,1,122,10.413800,122,10.413800,61.927502,122,10.413800,61.927502
1,1,담양군(금월교),Damyanggun(Geumwolgyo),5001615,500103,200001,202402,35.332778,127.016667,104,0.000417,127.016821,35.333165,68.325275,0.082241,103,0.965819,2,6,2,57,3.949320,57,3.949320,319.764008,57,3.949320,319.764008
2,2,담양군(덕용교),Damyanggun(Deokyonggyo),5001620,500102,197301,202402,35.289444,127.039167,97,0.000270,127.039104,35.289182,-103.485963,0.100323,96,0.553579,3,24,3,55,0.991447,55,0.991447,231.339996,55,0.991447,231.339996
3,3,담양군(삼지교),Damyanggun(Samjigyo),5001625,500103,200308,202402,35.270556,126.935833,98,0.001229,126.936319,35.269427,-66.743353,0.215554,97,0.405881,4,23,4,30,0.346259,30,0.346259,333.434998,30,0.346259,333.434998
4,4,담양군(양지교),Damyanggun(Yangjigyo),5001627,500105,201210,202402,35.256111,126.943611,31,0.000705,126.943736,35.255417,-79.765661,0.250858,30,0.231006,5,25,5,31,0.437983,31,0.437983,315.000000,31,0.437983,315.000000
5,5,광주광역시(용산교),Gwangju(Yongsangyo),5001640,500106,199107,202402,35.240556,126.888611,91,0.001276,126.887686,35.241435,136.461789,0.040490,90,0.896848,6,38,6,24,0.734494,24,0.734494,341.565002,24,0.734494,341.565002
6,6,광주광역시(유촌교),Gwangju(Yuchongyo),5001650,500107,200102,202402,35.166944,126.856667,39,0.001062,126.856381,35.167967,105.621946,0.136632,38,0.272890,7,49,7,22,0.734494,22,0.734494,251.565002,22,0.734494,251.565002
7,7,광주광역시(풍영정천2교),Gwangju(Pungyeongjeongcheon2gyo),5001655,500108,200712,202402,35.171667,126.814722,24,0.000543,126.814312,35.171311,-139.069678,0.110615,23,0.620529,8,42,8,22,2.191620,22,2.191620,47.862400,22,2.191620,47.862400
8,8,광주광역시(어등대교),Gwangju(Eodeungdaegyo),5001660,500108,200601,202402,35.160000,126.823056,109,0.000353,126.823400,35.160074,12.043886,0.152872,108,0.904045,9,54,9,14,1.336350,14,1.336350,79.991997,14,1.336350,79.991997
9,9,광주광역시(설월교),Gwangju(Seolwolgyo),5001670,500107,200409,202402,35.129444,126.927500,39,0.000275,126.927537,35.129717,82.243020,0.225249,38,0.984155,10,49,10,60,0.855143,60,0.855143,5.194430,60,0.855143,5.194430


In [6]:
# Define the CSV file path (change or keep it as you wish)
csv_file_path = r"C:\data_gnn_ver2\sites_with_dem_properties.csv"
# Export DataFrame to CSV
df.to_csv(csv_file_path, encoding="utf-8-sig", index=False)

print("Process completed. The CSV file has been saved to:", csv_file_path)

# Check in the Spatial Analyst extension
arcpy.CheckInExtension("Spatial")

Process completed. The CSV file has been saved to: C:\data_gnn_ver2\sites_with_dem_properties.csv


'CheckedOut'

## Adjacency Matrix

In [3]:
import arcpy
import arcgisscripting, sys, string, os, re, math
from time import *
from decimal import Decimal
import pandas as pd
from collections import defaultdict, deque
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from arcpy.sa import *

### 1. Generate the (node) relationship table

#### 1-1. Polyline to Landscape Network

In [4]:
arcpy.env.workspace = r"C:\TEST3"
arcpy.env.overwriteOutput = True

Reachfeatureclass = r"C:\TEST3\line_ys\line_ys.shp" # 절대 이름을 edges.shp이라고 짓지 말 것.
geoDatabasePath = r"C:\TEST3\FINAL_OUTPUT\FINAL_OUTPUT2.gdb"

In [5]:
# Create the Geoprocessor object
gp = arcgisscripting.create()


def LookForEdge(ToFeatGT, FromFeatGT, NewFromFeatList, NewToFeatList, FID):
    exists = FID in ToFeatGT
    if exists == 1:
        while exists == 1:
            #gp.AddMessage("  " + str(FID))
            ind = ToFeatGT.index(FID)
            exists2 = FromFeatGT[ind] in NewToFeatList
            if exists2 == 1:

                ind2 = NewToFeatList.index(FromFeatGT[ind])
                NewFromFeatList.insert((ind2), FromFeatGT[ind])
                NewToFeatList.insert((ind2), ToFeatGT[ind])
            else:

                NewFromFeatList.append(FromFeatGT[ind])
                NewToFeatList.append(ToFeatGT[ind])

            del FromFeatGT[ind]
            del ToFeatGT[ind]
            exists = FID in ToFeatGT

    return 1

def uniqueFeature(unique_feature_list, fid):
    if fid not in unique_feature_list:
        unique_feature_list.append(fid)
    return 1

#Main Function
if __name__ == "__main__":

#     Reachfeatureclass = sys.argv[1]
#     geoDatabasePath = sys.argv[2]

    # Extract the directory path and geodatabase name from the provided path
    gdbDirectory = os.path.dirname(geoDatabasePath)
    gdbName = os.path.basename(geoDatabasePath)
    
    # Verify if the shapefile exists
    if not arcpy.Exists(Reachfeatureclass):
        raise OSError(f"{Reachfeatureclass} does not exist")

    # Create File Geodatabase
    arcpy.AddMessage("Creating File Geodatabase " + gdbName)
    arcpy.CreateFileGDB_management(gdbDirectory, gdbName)

    # Adjust path for the new File Geodatabase
    geoDatabase = os.path.join(gdbDirectory, gdbName)

    # Create tables in the Geodatabase
    arcpy.CreateTable_management(geoDatabase, "nodexy")
    arcpy.AddField_management(geoDatabase + "/nodexy", "pointid", "LONG", 9)
    arcpy.AddField_management(geoDatabase + "/nodexy", "xcoord", "DOUBLE", 12)
    arcpy.AddField_management(geoDatabase + "/nodexy", "ycoord", "DOUBLE", 12)

    gp.AddMessage(" ")
    gp.AddMessage("    Creating Relationship Tables....")
    gp.AddMessage(" ")

    # Create blank table in Geodatabase to be filled with node edge relationsships.
    gp.CreateTable(geoDatabase, "noderelationships")
    gp.AddField(geoDatabase + "/noderelationships", "rid", "long", "9")
    gp.AddField(geoDatabase + "/noderelationships", "fromnode", "long", "9")
    gp.AddField(geoDatabase + "/noderelationships", "tonode", "long", "9")

    # Create relationships table that holds feature to feature relationships
    gp.CreateTable(geoDatabase, "relationships")
    gp.AddField(geoDatabase + "/relationships", "fromfeat", "long", "9")
    gp.AddField(geoDatabase + "/relationships", "tofeat", "long", "9")

    #Loop through the table using a cursor
    rows = gp.searchCursor(Reachfeatureclass)
    row = rows.Next()

    #Set up Insert cursor for point x y table
    Nodexyrows = gp.InsertCursor(geoDatabase + "/nodexy")
    Nodexyrow = Nodexyrows.NewRow()

    gp.AddMessage("  ")
    gp.AddMessage("Building Hydro Relationships ...")
    gp.AddMessage("  ")
    PntID = 0
    while row:
        # Get the class (feature class) for that row, as well as the Feature ID
        FID = row.GetValue("fid")
        #gp.AddMessage(str(FID))
        feature = row.shape
        FromPoint = feature.FirstPoint
        ToPoint = feature.LastPoint

        if PntID == 0:
            PntIDList = [PntID]
            CoordList = [FromPoint]
            FromPointIDList = [PntID]
            coords = FromPoint.split(" ")
            Nodexyrow.pointid = PntID
            Nodexyrow.xcoord = coords[0]
            Nodexyrow.ycoord = coords[1]
            Nodexyrows.InsertRow(Nodexyrow)
            PntID = PntID + 1
            PntIDlist = PntIDList.append(PntID)
            CoordList.append(ToPoint)
            FeatureList = [FID]
            ToPointIDList = [PntID]
            coords = ToPoint.split(" ")
            Nodexyrow.pointid = PntID
            Nodexyrow.xcoord = coords[0]
            Nodexyrow.ycoord = coords[1]
            Nodexyrows.InsertRow(Nodexyrow)
        else:
            fromInd = FromPoint in CoordList        # returns 1 if exists 0 if not
            toInd = ToPoint in CoordList            # returns 1 if exists 0 if not
            FeatureList.append(FID)
            if fromInd > 0 and toInd == 0:          #Frompoint coord exists in Coordlist and ToPoint doesn't, so add ToPoint to CoordList
                #print "Adding ToPoint Coordinates"
                PntIDList.append(PntID)
                CoordList.append(ToPoint)
                ind = CoordList.index(FromPoint)
                PointID = PntIDList[ind]
                FromPointIDList.append(PointID)
                ToPointIDList.append(PntID)
                # Insert into table
                coords = ToPoint.split(" ")
                Nodexyrow.pointid = PntID
                Nodexyrow.xcoord = coords[0]
                Nodexyrow.ycoord = coords[1]
                Nodexyrows.InsertRow(Nodexyrow)

            elif fromInd == 0 and toInd > 0:
                #print "Adding FromPoint Coordinates"
                PntIDList.append(PntID)
                CoordList.append(FromPoint)
                ind = CoordList.index(ToPoint)
                #print ind
                PointID = PntIDList[ind]
                FromPointIDList.append(PntID)
                ToPointIDList.append(PointID)
                # Insert into table
                coords = FromPoint.split(" ")
                Nodexyrow.pointid = PntID
                Nodexyrow.xcoord = coords[0]
                Nodexyrow.ycoord = coords[1]
                Nodexyrows.InsertRow(Nodexyrow)

            elif fromInd == 0 and toInd == 0:
                #print "Adding both coordinates"
                PntIDList.append(PntID)
                CoordList.append(FromPoint)
                FromPointIDList.append(PntID)
                # Insert into table
                coords = FromPoint.split(" ")
                Nodexyrow.pointid = PntID
                Nodexyrow.xcoord = coords[0]
                Nodexyrow.ycoord = coords[1]
                Nodexyrows.InsertRow(Nodexyrow)
                PntID = PntID + 1
                PntIDList.append(PntID)
                CoordList.append(ToPoint)
                ToPointIDList.append(PntID)
                # Insert into table
                coords = ToPoint.split(" ")
                Nodexyrow.pointid = PntID
                Nodexyrow.xcoord = coords[0]
                Nodexyrow.ycoord = coords[1]
                Nodexyrows.InsertRow(Nodexyrow)

            elif fromInd > 0 and toInd > 0:
                #print "Not adding coordinates"
                ind = CoordList.index(ToPoint)
                PointID = PntIDList[ind]
                ToPointIDList.append(PointID)
                ind = CoordList.index(FromPoint)
                PointID = PntIDList[ind]
                FromPointIDList.append(PointID)

        PntID = PntID + 1
        row = rows.Next()
        Nodexyrow = Nodexyrows.NewRow()

    #Setup Insert cursor to populate index table with where a features relationships are located in the relationships table
    # Setup Insert cursor to populate relationships table with feature to feature relationships
    featureRelrows = gp.InsertCursor(geoDatabase + "/relationships")
    startIndex = 0 # this is used to get the FID in the main loop
    GTfound = 0

    #Set up Inset cursor for noderelationship table
    Noderows = gp.InsertCursor(geoDatabase + "/noderelationships")

    # build two lists one for from features and one for to features these two list will be sorted downsteam.  The SinkLIst contains
    # all features that are sinks or have no to feature defined.  This list will be used to sort the other features downstream
    nodeIDindex = 0
    edgeID = 0
    gp.AddMessage(" ")
    gp.AddMessage("    Building Landscape Network Relationships....")
    gp.AddMessage(" ")
    FromFeatGT = [] # list of from Features
    ToFeatGT = [] # list of to features is matches the from feature list
    uniqueFeatureList = [] # list on unique feature ids this is used to find sink features
    for ToFeature in ToPointIDList:
        # populate noderelationships table
        Noderow = Noderows.NewRow()
        Noderow.rid = edgeID
        dummy = uniqueFeature(uniqueFeatureList, edgeID) # this function queries if this feature exists in list if it doesn't add id
        Noderow.fromnode = FromPointIDList[nodeIDindex]
        Noderow.tonode = ToFeature
        Noderows.InsertRow(Noderow)
        forwardIndex  = 0 # this is the location in the list that fromnode and tondoe are equal, and is used to get the coresponding FID
        for FromFeature in FromPointIDList:
            if FromFeature == ToFeature:
                FromFeatGT.append(FeatureList[startIndex])
                ToFeatGT.append(FeatureList[forwardIndex])
            forwardIndex = forwardIndex + 1
        nodeIDindex = nodeIDindex + 1
        edgeID = edgeID + 1
        startIndex = startIndex + 1

    # this loop get all sink features in edge feature class to be used to calculate up stream edges
    SinkList = [] # list of all sink features
    for FID in uniqueFeatureList:
        exists = FID in FromFeatGT
        if exists == 0:
            SinkList.append(FID)

    uniqueFeatureList = []  # set uniqueFeatureList to nothing
    gp.AddMessage(" ")
    gp.AddMessage("    Sorting Relationship Table Downstream....")
    gp.AddMessage(" ")
    # this loop start a all Sink features in the SinkList and builds to from list working up stream to build a sorted downstream list
    NewFromFeatList = [] # this is a new list of from features sorted upstream
    NewToFeatList = [] # this is a new list of to features sortes upstream
    sinkcount = 1
    for SinkFID in SinkList:

        exists = SinkFID in ToFeatGT
        if exists == 1:

            if sinkcount == 1:
                while exists == 1:
                    ind = ToFeatGT.index(SinkFID)

                    NewFromFeatList.append(FromFeatGT[ind])
                    NewToFeatList.append(ToFeatGT[ind])
                    del FromFeatGT[ind]
                    del ToFeatGT[ind]
                    exists = SinkFID in ToFeatGT
                for FID in NewFromFeatList:

                    dummy = LookForEdge(ToFeatGT, FromFeatGT, NewFromFeatList, NewToFeatList, FID) # this function gets upstream edges from a given FID
            else:
                tempFrom = []
                tempTo = []
                count = 1
                while exists == 1:

                    ind = ToFeatGT.index(SinkFID)

                    tempFrom.append(FromFeatGT[ind])
                    tempTo.append(ToFeatGT[ind])
                    del FromFeatGT[ind]
                    del ToFeatGT[ind]
                    exists = SinkFID in ToFeatGT
                    count = count + 1
                for FID in tempFrom:

                    dummy = LookForEdge(ToFeatGT, FromFeatGT, tempFrom, tempTo, FID) # this function gets upstream edges from a given FID

            if sinkcount != 1:
                lastIndex = (len(tempFrom) - 1)
                exists = tempFrom[lastIndex] in NewToFeatList
                exists2 = tempFrom[lastIndex] in NewFromFeatList
                if exists == 1:

                    x = len(tempFrom) - 1
                    while x > -1:
                        exists = tempFrom[x] in NewToFeatList
                        if exists == 1: # this will place stream reaches in correct downstream order
                            index = NewToFeatList.index(tempFrom[x])
                        NewFromFeatList.insert(index, tempFrom[x])
                        NewToFeatList.insert(index, tempTo[x])
                        x = x - 1
                elif exists2 == 1:
                    x = len(tempFrom) - 1
                    while x > -1:
                        exists = tempFrom[x] in NewFromFeatList
                        if exists == 1: # this will place stream reaches in correct downstream order
                            index = NewFromFeatList.index(tempFrom[x])
                        else:
                            index = 0
                        NewFromFeatList.insert(index, tempFrom[x])
                        NewToFeatList.insert(index, tempTo[x])
                        x = x - 1
                else:
                    indexcount = 0
                    for fid in tempFrom:
                        NewFromFeatList.append(fid)
                        NewToFeatList.append(tempTo[indexcount])
                        indexcount = indexcount + 1

        sinkcount = sinkcount + 1
    tempFrom = "nothing"
    tempTo = "nothing"

    # go through NewFeatLists from the bottom up to sort downstream
    x = len(NewFromFeatList) - 1
    while x > -1:
        featureRelrow = featureRelrows.NewRow()
        featureRelrow.fromfeat = NewFromFeatList[x]
        featureRelrow.tofeat = NewToFeatList[x]
        featureRelrows.InsertRow(featureRelrow)
        x = x - 1
    gp.AddMessage(" ")
    gp.AddMessage("Creating Landscape Network Features Classes....")
    gp.AddMessage(" ")

    desc = gp.Describe(Reachfeatureclass)

    #Create point layer from x,y table
    gp.workspace = geoDatabase
    # make Event Layer out of x,y table - ADD SPATIAL REFERENCE
    gp.MakeXYEventLayer(geoDatabase + "/nodexy", "xcoord" , "ycoord", "nodes.lyr", desc.spatialReference)
    # create a temp. featurelayer out of layer file
    gp.MakeFeatureLayer("nodes.lyr","nodes")

    # Copy Nodes, Reach input, and hCA layers into new featureclasses in GeoDataBase
    gp.CopyFeatures_management("nodes.lyr", geoDatabase + "/nodes")
    
    # Copy the reaches shape file into the geodatabase
    gp.CopyFeatures(Reachfeatureclass, geoDatabase + "/edges")

    gp.Workspace = geoDatabase

    # Add the field RID and calculate it to equal the OID field
    if not gp.ListFields("edges", "rid").Next():
        gp.AddField("edges", "rid", "LONG", 12, 12)
    if gp.ListFields("edges", "*", "OID").Next():
        OIDField = gp.ListFields("edges", "*", "OID").Next()
        # Updated the CalculateField syntax
        expression = f"!{OIDField.Name}! - 1"
        gp.CalculateField_management("edges", "rid", expression, "PYTHON3")

    gp.AddMessage(" ")
    gp.AddWarning("FINISHED Polyline to Landscape Network Script")
    gp.AddMessage(" ")
    gp.AddMessage(" ")
    gp.AddMessage(" ")

#### 1-2. Identify Complex Confluences

In [6]:
arcpy.env.workspace = r"C:\TEST3"
arcpy.env.overwriteOutput = True

outputFileName = r"C:\TEST3\id_errors.txt"
lsnWorkspace = r"C:\TEST3\FINAL_OUTPUT\FINAL_OUTPUT2.gdb"

# Set the workspace for LSN
arcpy.env.workspace = lsnWorkspace

# Open the output file
with open(outputFileName, "w") as ofh:
    node_string = ","

    try:
        # Order the noderelationships table by TONODE and create a table view
        qry = '1=1 ORDER BY "TONODE"'
        arcpy.MakeTableView_management("noderelationships", "temptable", qry)

        arcpy.AddMessage("Creating SearchCursor")

        oldValue = -99
        nodeList = []
        count = 1

        arcpy.AddMessage("Looking for loops")
        with arcpy.da.SearchCursor("temptable", ["TONODE"]) as rows:
            for row in rows:
                newValue = row[0]  # Directly getting the value from the cursor

                if newValue == oldValue:
                    arcpy.AddMessage(f"newValue {newValue} = oldValue = {oldValue}")
                    nodeList.append(newValue)
                    count += 1
                else:
                    arcpy.AddMessage(f"newValue {newValue} <> oldValue = {oldValue}")
                    oldValue = newValue
                    count = 1

                if count > 2:
                    node_string += str(newValue) + ", "

        arcpy.AddMessage(node_string)
        ofh.write(node_string)

        arcpy.AddWarning("Program finished successfully")

    except Exception as e:
        arcpy.AddError(f"Error: {str(e)}")
        arcpy.AddError(arcpy.GetMessages())


#### 1-3. Snap Points to Landscape Network

In [7]:
arcpy.env.workspace = r"C:\TEST3"
arcpy.env.overwriteOutput = True

# Input Sample Points and Edge Network Feature classes
SamplePTS = r"C:\TEST3\obs_ys\obs_stations.shp"  # Sample points to snap to network
EdgeNetwork = r"C:\TEST3\FINAL_OUTPUT\FINAL_OUTPUT2.gdb\edges"  # Network to snap sample points to
OutPutFC = r"C:\TEST3\FINAL_OUTPUT\FINAL_OUTPUT2.gdb\sites"  # Output snapped points name and location -> this should be a feature class in a PGDB
SearchLength = "1"  # max search distance
TempWorkspace = r"C:\Temp"  # temporary workspace

# Create the Geoprocessor object
arcpy.env.overwriteOutput = True

# This function calculates distance between points
def CalcDist(startx, starty, tox, toy):
    a = (starty - toy)
    b = (startx - tox)
    dist = math.hypot(a, b)
    return dist

# This function reselects an edge with a fid equal to fid and breaks it up into parts
def IsOnLine(fromx, fromy, tox, toy, pntx, pnty):
    r = CalcDist(fromx, fromy, tox, toy)  # distance of segment between to and from vertices
    rprime = CalcDist(fromx, fromy, pntx, pnty)  # distance between from vertex and new point coords

    if rprime == 0:
        return 1
    if abs(rprime - r) < 0.001:
        return 1
    if rprime > r:
        return 0

    ydiff = abs(fromy - toy)
    yprimediff = abs(fromy - pnty)

    ratio = round(math.sin(ydiff / r), 2)
    ratioprime = round(math.sin(yprimediff / rprime), 2)
    if ratio == ratioprime:
        return 1
    else:
        return 0

def DynamicSplit3(fid, xcoord, ycoord, edgeFCName):
    with arcpy.da.SearchCursor(edgeFCName, ["SHAPE@"], f"rid = {fid}") as rows:
        for row in rows:
            feature = row[0]
            fLength = feature.length
            totaldist = 0
            pointdist = 0
            fromx = fromy = None
            mindist = 999999
            for part in feature:
                for i, pnt in enumerate(part):
                    if fromx is not None and fromy is not None:
                        dist2 = CalcDist(fromx, fromy, pnt.X, pnt.Y)
                        dist1 = CalcDist(xcoord, ycoord, pnt.X, pnt.Y)
                        if dist1 < mindist:
                            if IsOnLine(fromx, fromy, pnt.X, pnt.Y, xcoord, ycoord) == 1:
                                mindist = dist1
                                fromdist = CalcDist(fromx, fromy, xcoord, ycoord)
                                todist = CalcDist(pnt.X, pnt.Y, xcoord, ycoord)
                                if fromdist < todist:
                                    pointdist = totaldist + fromdist
                                else:
                                    pointdist = totaldist + (dist2 - todist)
                    else:
                        if IsOnLine(pnt.X, pnt.Y, pnt.X, pnt.Y, xcoord, ycoord) == 1:
                            dist1 = CalcDist(xcoord, ycoord, pnt.X, pnt.Y)
                            totaldist = dist1
                            pointdist = totaldist
                        dist2 = 0
                    totaldist += dist2
                    fromx, fromy = pnt.X, pnt.Y

            length = (fLength - pointdist)
            if fLength == 0:
                ratio = 1
            else:
                ratio = float(length / fLength)
            if ratio < .0001:
                ratio = .001
            return ratio

if __name__ == "__main__":
    try:
        samplePTS = os.path.join(arcpy.Describe(SamplePTS).path, arcpy.Describe(SamplePTS).name)
        EdgeNetwork = os.path.join(arcpy.Describe(EdgeNetwork).path, arcpy.Describe(EdgeNetwork).name)
        PGDBPath = arcpy.Describe(EdgeNetwork).path
        SHPWorkspace = arcpy.Describe(SamplePTS).path
        SampleFCName = arcpy.Describe(SamplePTS).name
        edgesFCName = arcpy.Describe(EdgeNetwork).name
        outputFCName = os.path.basename(OutPutFC)

        arcpy.env.workspace = PGDBPath

        # Obtain spatial reference to define projection for output data
        spatialRef = arcpy.Describe(samplePTS).spatialReference

        # Run the NEAR command to find the closest edge
        arcpy.AddMessage("\nRunning NEAR Command...\n")
        arcpy.analysis.Near(SamplePTS, EdgeNetwork, SearchLength, "LOCATION", "ANGLE")
        arcpy.AddMessage("Evaluating edges...\n")

        # Create a temporary shapefile
        tempSHP = os.path.join(TempWorkspace, "snaptemp.shp")
        if arcpy.Exists(tempSHP):
            arcpy.Delete_management(tempSHP)
        arcpy.CreateFeatureclass_management(os.path.dirname(tempSHP), os.path.basename(tempSHP), "POINT", SamplePTS, spatial_reference=spatialRef)

        # Check if necessary fields exist
        if not arcpy.ListFields(tempSHP, "rid"):
            arcpy.AddField_management(tempSHP, "rid", "LONG")
        if not arcpy.ListFields(tempSHP, "ratio"):
            arcpy.AddField_management(tempSHP, "ratio", "DOUBLE")

        with arcpy.da.InsertCursor(tempSHP, ["SHAPE@", "rid", "ratio"] + [f.name for f in arcpy.ListFields(SamplePTS) if f.type != 'Geometry']) as pointCur:
            with arcpy.da.SearchCursor(SamplePTS, ["NEAR_FID", "NEAR_X", "NEAR_Y"] + [f.name for f in arcpy.ListFields(SamplePTS) if f.type != 'Geometry']) as rows:
                arcpy.AddMessage("Creating Snapped Points Featureclass...\n")

                for row in rows:
                    FID, xcoord, ycoord = row[:3]
                    attributes = row[3:]
                    if FID != -1:
                        FID = FID - 1
                        ratio = DynamicSplit3(FID, xcoord, ycoord, EdgeNetwork)
                        pointCur.insertRow([arcpy.Point(xcoord, ycoord), FID, ratio] + list(attributes))

        # Define projection for output points
        arcpy.DefineProjection_management(tempSHP, spatialRef)

        # Copy the temporary shapefile to the output feature class
        if arcpy.Exists(OutPutFC):
            arcpy.Delete_management(OutPutFC)
        arcpy.FeatureClassToFeatureClass_conversion(tempSHP, os.path.dirname(OutPutFC), os.path.basename(OutPutFC))

        arcpy.Delete_management(tempSHP)
        arcpy.AddWarning("\n\nFinished Snap Points to Landscape Network Edges\n\n")
        print("Finished Snap Points to Landscape Network Edges")
    except Exception as e:
        arcpy.AddError("Did NOT Snap Points to Landscape Network Edges")
        arcpy.AddError(f"Error: {str(e)}")
        arcpy.AddError(arcpy.GetMessages())
        if 'tempSHP' in locals() and arcpy.Exists(tempSHP):
            arcpy.Delete_management(tempSHP)


Finished Snap Points to Landscape Network Edges


#### 1-4. edges에서 Field Name column 속 OBJECTID를 삭제 후, OBJECTID_1를 OBJECTID로 수정

In [8]:
# Set the workspace
arcpy.env.workspace = r"C:\TEST3\FINAL_OUTPUT\FINAL_OUTPUT2.gdb"
edges_shapefile = "edges"

# Check if the "OBJECTID" field exists and delete it
if arcpy.ListFields(edges_shapefile, "OBJECTID"):
    arcpy.DeleteField_management(edges_shapefile, "OBJECTID")
    arcpy.AddMessage(f"Field 'OBJECTID' deleted from {edges_shapefile}")

# Check if the "OBJECTID_1" field exists and rename its alias to "OBJECTID"
fields = arcpy.ListFields(edges_shapefile, "OBJECTID_1")
if fields:
    field = fields[0]
    if field.aliasName != "OBJECTID":
        arcpy.AlterField_management(edges_shapefile, "OBJECTID_1", new_field_alias="OBJECTID")
        arcpy.AddMessage(f"Field alias for 'OBJECTID_1' renamed to 'OBJECTID' in {edges_shapefile}")

arcpy.AddMessage("Field operations completed successfully.")


#### 1-5. Upstream Distance - Edges

In [9]:
edgesFC = r"C:\TEST3\FINAL_OUTPUT\FINAL_OUTPUT2.gdb\edges"
LengthField = "Shape_Length"


try:
    # Set the necessary product code
    arcpy.SetProduct("ArcInfo")

    # Check out any necessary licenses
    arcpy.CheckOutExtension("Network")

    arcpy.AddMessage(f"edgesFC: {edgesFC}")
    arcpy.AddMessage(f"LengthField: {LengthField}")

    # Local variables
    gdbPath = os.path.dirname(edgesFC)
    RelTableName = os.path.join(gdbPath, "relationships")

    # Check if the feature class and relationship table exist
    if not arcpy.Exists(edgesFC):
        arcpy.AddError(f"Error: {edgesFC} does not exist.")
        sys.exit(1)

    if not arcpy.Exists(RelTableName):
        arcpy.AddWarning("Relationship table doesn't exist")
        sys.exit(1)

    arcpy.AddMessage("Relationship table exists")

    # Initialize lists for accumulation
    FeatureList = []
    AccumulateValueList = []

    arcpy.AddMessage("Accumulating Upstream...")

    # Iterate through each relationship
    with arcpy.da.SearchCursor(RelTableName, ['fromfeat', 'tofeat']) as cursor:
        for row in cursor:
            fromfeat, tofeat = row

            arcpy.AddMessage(f"Processing relationship from {fromfeat} to {tofeat}")

            # Fetch values from edgesFC
            fromvalue = 0
            tovalue = 0

            # Get length value for 'fromfeat'
            where_clause = f"rid = {fromfeat}"
            with arcpy.da.SearchCursor(edgesFC, [LengthField], where_clause) as from_cursor:
                for from_row in from_cursor:
                    fromvalue += from_row[0]

            # Get length value for 'tofeat'
            where_clause = f"rid = {tofeat}"
            with arcpy.da.SearchCursor(edgesFC, [LengthField], where_clause) as to_cursor:
                for to_row in to_cursor:
                    tovalue += to_row[0]

            # Accumulation logic
            toexists = tofeat in FeatureList
            fromexists = fromfeat in FeatureList
            if not fromexists:  # if fromfeature not in list add it and add its weight value to accumulate list
                FeatureList.append(fromfeat)
                AccumulateValueList.append(fromvalue)

            if toexists:  # if tofeature exists in list accumulate it
                ind = FeatureList.index(tofeat)
                if fromexists:  # if both fromfeature and tofeature exist in list add fromfeature's list value to to node value
                    ind2 = FeatureList.index(fromfeat)
                    AccumulateValueList[ind] += AccumulateValueList[ind2]
                else:
                    AccumulateValueList[ind] += fromvalue
            else:
                FeatureList.append(tofeat)
                AccumulateValueList.append(tovalue + fromvalue)

    # Add or update the 'upDist' field
    if arcpy.ListFields(edgesFC, "upDist"):
        arcpy.AddMessage("Updating Field upDist...")
    else:
        arcpy.AddMessage("Adding Field upDist...")
        arcpy.AddField_management(edgesFC, "upDist", "DOUBLE")

    # Update the 'upDist' field in edgesFC
    with arcpy.da.UpdateCursor(edgesFC, ['rid', 'upDist']) as cursor:
        for row in cursor:
            rid = row[0]
            if rid in FeatureList:
                ind = FeatureList.index(rid)
                row[1] = AccumulateValueList[ind]
                cursor.updateRow(row)

    arcpy.AddMessage("Upstream accumulation completed successfully.")

except Exception as e:
    arcpy.AddError(f"An error occurred: {e}")
    arcpy.AddWarning("Program DID NOT finish successfully")



#### 1-6. Upstream Distance - Sites

In [10]:
# Input parameters from command line or script tool
edgesFC =  r"C:\TEST3\FINAL_OUTPUT\FINAL_OUTPUT2.gdb\edges"
LengthField = "Shape_Length"
sitesFCString = r"C:\TEST3\FINAL_OUTPUT\FINAL_OUTPUT2.gdb\sites"


try:

    arcpy.AddMessage("LengthField: " + str(LengthField))

    # Split the sites feature class string into a list
    sitesFCList = sitesFCString.split(';')

    # Set the workspace to the path of the edges feature class
    workspace = arcpy.Describe(edgesFC).path
    arcpy.env.workspace = workspace

    # Make a feature layer from the edges feature class
    edgesFCName = arcpy.Describe(edgesFC).name
    arcpy.MakeFeatureLayer_management(edgesFCName, "edgeLyr")

    # Process each sites feature class
    for SitesFCName in sitesFCList:
        arcpy.MakeFeatureLayer_management(SitesFCName, "siteLyr")

        # Add "upDist" field if it does not exist
        if len(arcpy.ListFields("siteLyr", "upDist")) == 0:
            arcpy.AddField_management("siteLyr", "upDist", "DOUBLE")

        arcpy.AddMessage("Calculating upstream distance for sites in " + SitesFCName)

        with arcpy.da.UpdateCursor("siteLyr", ["rid", "ratio", "upDist"]) as siteRows:
            for siteRow in siteRows:
                siteRID, siteRatio = siteRow[0], siteRow[1]
                arcpy.AddMessage(f"Processing site RID: {siteRID} with ratio: {siteRatio}")

                query = "rid = {}".format(siteRID)
                with arcpy.da.SearchCursor("edgeLyr", ["rid", "upDist", LengthField], query) as edgeRows:

                    edgeFound = False
                    for edgeRow in edgeRows:
                        edgeFound = True
                        accAttribute, scaAttribute = edgeRow[1], edgeRow[2]
                        h20att = accAttribute - ((1 - siteRatio) * scaAttribute)

                        arcpy.AddMessage(f"Edge found for site RID {siteRID}. Updated upDist: {h20att}")
                        siteRow[2] = h20att
                        siteRows.updateRow(siteRow)

                    if not edgeFound:
                        arcpy.AddMessage(f"No matching edge found for site RID {siteRID}")


        arcpy.Delete_management("siteLyr")

    arcpy.Delete_management("edgeLyr")

    arcpy.AddMessage("Program finished successfully")

except Exception as e:
    arcpy.AddError("Error occurred: " + str(e))
    arcpy.AddMessage(arcpy.GetMessages())

### 2. Split stream edges by obs station shp

In [11]:
# Set the workspace
arcpy.env.workspace = r"C:\TEST3\FINAL_OUTPUT\FINAL_OUTPUT2.gdb"
arcpy.env.overwriteOutput = True

# Define the input feature classes
points_fc = r"C:\TEST3\FINAL_OUTPUT\FINAL_OUTPUT2.gdb\sites"
lines_fc = r"C:\TEST3\FINAL_OUTPUT\FINAL_OUTPUT2.gdb\edges"

# Specify the output feature class for the split lines
split_lines_fc = r"C:\TEST3\FINAL_OUTPUT\FINAL_OUTPUT2.gdb\split_edges"

# Define the search radius (optional)
search_radius = "30 Meters"  # Adjust this based on your needs

# Split the lines at the points
arcpy.management.SplitLineAtPoint(in_features=lines_fc, point_features=points_fc, 
                                  out_feature_class=split_lines_fc, search_radius=search_radius)

print("Process completed: The lines have been split at the points' locations.")


Process completed: The lines have been split at the points' locations.


### 3. Add new_length variable [km] with split stream edges

In [12]:
split_lines_fc = r"C:\TEST3\FINAL_OUTPUT\FINAL_OUTPUT2.gdb\split_edges"

# Add a new field to store the length of each split line
new_field_name = "new_length"
arcpy.AddField_management(in_table=split_lines_fc, field_name=new_field_name, field_type="DOUBLE")

# Calculate the length of each line and store it in the new field
# The calculation will depend on the coordinate system of your data. 
# For geographic coordinate systems, use GEODESIC, GREAT_ELLIPTIC, LOXODROME, or PRESERVE_SHAPE
# For projected coordinate systems, you can use PLANAR (default).
length_type = "GEODESIC"  # Adjust this based on your data's coordinate system

arcpy.CalculateGeometryAttributes_management(in_features=split_lines_fc, 
                                             geometry_property=[(new_field_name, f"LENGTH_{length_type}")],
                                             length_unit="KILOMETERS")  # Adjust the length unit as necessary

print("New length column added and calculated for each split line segment.")

New length column added and calculated for each split line segment.


### 4. Add point shp's attributes to split stream edges

In [13]:
# Specify the output feature class for the joined results
joined_split_lines_fc = r"C:\TEST3\FINAL_OUTPUT\FINAL_OUTPUT2.gdb\joined_split_edges"

# Perform a spatial join to transfer the point attributes to the split lines
arcpy.analysis.SpatialJoin(target_features=split_lines_fc, join_features=points_fc,
                           out_feature_class=joined_split_lines_fc, join_operation="JOIN_ONE_TO_ONE",
                           match_option="CLOSEST", search_radius=search_radius)

print("Spatial join completed. Point attributes transferred to split lines.")

# Add a new field to store the length of each joined split line
new_joined_field_name = "new_length"
arcpy.AddField_management(in_table=joined_split_lines_fc, field_name=new_joined_field_name, field_type="DOUBLE")

# Calculate the length of each joined split line and store it in the new field
arcpy.CalculateGeometryAttributes_management(in_features=joined_split_lines_fc, 
                                             geometry_property=[(new_joined_field_name, f"LENGTH_{length_type}")],
                                             length_unit="KILOMETERS")  # Adjust the length unit as necessary

print("New length column added and calculated for each joined split line segment.")

Spatial join completed. Point attributes transferred to split lines.
New length column added and calculated for each joined split line segment.


### 5. Make GDB to dataframe

In [14]:
# Function to convert a feature class or table to pandas DataFrame
def gdb_to_df(path, feature_class):
    # List to hold data
    data = []
    # Full path to the feature class or table
    full_path = f"{path}\\{feature_class}"
    # Fields to extract (adjust as necessary)
    fields = [f.name for f in arcpy.ListFields(full_path) if f.type != 'Geometry']
    # Cursor to read data
    with arcpy.da.SearchCursor(full_path, fields) as cursor:
        for row in cursor:
            data.append(row)
    # Convert to DataFrame
    df = pd.DataFrame(data, columns=fields)
    return df

In [15]:
# Set the workspace
arcpy.env.workspace = r"C:\TEST3\FINAL_OUTPUT\FINAL_OUTPUT2.gdb"
arcpy.env.overwriteOutput = True

# Define the feature classes and tables
gdb_path = r"C:\TEST3\FINAL_OUTPUT\FINAL_OUTPUT2.gdb"
points_fc = "sites"
lines_fc = "edges"
split_lines_fc = "joined_split_edges"
relationships_table = "relationships"
noderelationships_table = "noderelationships"
nodes_fc = "nodes"

# Convert feature classes and tables to DataFrames
sites_df = gdb_to_df(gdb_path, points_fc)
edges_df = gdb_to_df(gdb_path, lines_fc)
split_edges_df = gdb_to_df(gdb_path, split_lines_fc)
relationships_df = gdb_to_df(gdb_path, relationships_table)
noderelationships_df = gdb_to_df(gdb_path, noderelationships_table)
nodes_df = gdb_to_df(gdb_path, nodes_fc)

# Print the DataFrames to verify the data
print("Sites DataFrame:")
print(sites_df.head())

print("\nEdges DataFrame:")
print(edges_df.head())

print("\nSplit Edges DataFrame:")
print(split_edges_df.head())

print("\nRelationships DataFrame:")
print(relationships_df.head())

print("\nNode Relationships DataFrame:")
print(noderelationships_df.head())

print("\nNodes DataFrame:")
print(nodes_df.head())

# Get unique nodes from noderelationships_df
nodes = pd.unique(noderelationships_df[['fromnode', 'tonode']].values.ravel('K'))
print("\nUnique Nodes:")
print(nodes)

Sites DataFrame:
   OBJECTID Station_Na               Station_En  ...    upDist  rid     ratio
0         1        평림댐              PYUNGRIMDAM  ...  0.120817   18  0.734233
1         2   담양군(담양댐)   Damyanggun(Damyangdam)  ...  0.038413    5  0.932127
2         3   담양군(금월교)   Damyanggun(Geumwolgyo)  ...  0.082241  103  0.965819
3         4   담양군(덕용교)  Damyanggun(Deokyonggyo)  ...  0.100323   96  0.553579
4         5   담양군(삼지교)     Damyanggun(Samjigyo)  ...  0.215554   97  0.405881

[5 rows x 17 columns]

Edges DataFrame:
   OBJECTID_1    RCH_ID     RCH_DID  ...  Shape_Length  rid    upDist
0           1  50020103  5002010300  ...      0.047654    0  0.047654
1           2  50020214  5002021400  ...      0.020005    1  0.020005
2           3  50020213  5002021300  ...      0.008996    2  0.008996
3           4  50010305  5001030500  ...      0.055842    3  0.055842
4           5  50010306  5001030600  ...      0.062356    4  0.062356

[5 rows x 44 columns]

Split Edges DataFrame:
   OBJE

### 6. Compute the hydrologic distance

In [16]:
def compute_distance_matrix(sites_df, split_edges_df, relationships_df, noderelationships_df):
    # 각 사이트의 rid를 split_edges_df에서 해당하는 edge ID와 길이에 매핑하는 딕셔너리 생성
    site_to_edges = defaultdict(list)
    site_to_start_length = {}
    site_to_end_length = {}
    
    for _, row in sites_df.iterrows():
        # site의 rid 저장
        site_rid = row['rid']
        site_name = row['Station_Na']
        
        # split edges에서 site가 얹혀져 있는 edge id를 저장
        edge_ids = split_edges_df.loc[split_edges_df['rid'] == site_rid, 'rid'].values
        
        # site id를 key로 하는 edge id를 추가 (이미 있는 경우를 고려하여, extend)
        site_to_edges[site_rid].extend(edge_ids)
        
        # 시작 site의 split된 길이 계산
        orig_seq_3 = split_edges_df.loc[(split_edges_df['Station_Na'] == site_name) & (split_edges_df['ORIG_SEQ'] == 3), 'new_length'].values
        orig_seq_2 = split_edges_df.loc[(split_edges_df['Station_Na'] == site_name) & (split_edges_df['ORIG_SEQ'] == 2), 'new_length'].values
        
        if orig_seq_3.size > 0:
            site_to_start_length[site_rid] = orig_seq_3[0]
        elif orig_seq_2.size > 0:
            site_to_start_length[site_rid] = orig_seq_2[0]
        else:
            site_to_start_length[site_rid] = 0
        
        # 도착 site의 split된 길이 계산
        if orig_seq_3.size > 0 and orig_seq_2.size > 0:
            site_to_end_length[site_rid] = orig_seq_2[0] + orig_seq_3[0]
        elif orig_seq_3.size > 0:
            site_to_end_length[site_rid] = orig_seq_3[0]
        elif orig_seq_2.size > 0:
            site_to_end_length[site_rid] = orig_seq_2[0]
        else:
            site_to_end_length[site_rid] = 0
            
    # Create dictionaries to store the graph and node relationships
    graph = defaultdict(list)
    node_graph = defaultdict(list)

    # Build the graph from relationships_df
    for _, row in relationships_df.iterrows():
        from_edge = row['fromfeat']
        to_edge = row['tofeat']
        edge_length_values = split_edges_df.loc[split_edges_df['rid'] == to_edge, 'new_length'].values
        edge_length = edge_length_values[0] if edge_length_values.size > 0 else np.nan
        graph[from_edge].append((to_edge, edge_length))

    # Build the node graph from noderelationships_df
    for _, row in noderelationships_df.iterrows():
        edge_id = row['rid']
        edge_length_values = split_edges_df.loc[split_edges_df['rid'] == edge_id, 'new_length'].values
        edge_length = np.nansum(edge_length_values)  # Sum all elements, ignoring NaN values
        node_graph[edge_id].append((edge_id, edge_length))

    # station name을 인덱스와 열로 사용하여 empty 행렬 DataFrame 생성
    distance_matrix = pd.DataFrame(index=sites_df['Station_Na'], columns=sites_df['Station_Na'])

    # 각 site 쌍의 거리 계산
    for i, start_site in enumerate(sites_df['rid']):
        start_edges = site_to_edges[start_site]
        start_length = site_to_start_length[start_site]
        
        for j, end_site in enumerate(sites_df['rid']):
            if i == j:
                # 시작 site와 도착 site가 같으면 거리는 0
                distance_matrix.at[sites_df['Station_Na'].iloc[i], sites_df['Station_Na'].iloc[j]] = 0
                continue
            
            end_edges = site_to_edges[end_site]
            end_length = site_to_end_length[end_site]
            
            # BFS를 사용하여, 모든 시작 edge와 도착 edge 쌍 사이의 최소 거리를 찾는다.
            distance = min(bfs_distance(graph, node_graph, start_edge, end_edge) + start_length + end_length
                           for start_edge in start_edges for end_edge in end_edges)
            
            # 계산된 거리를 거리 행렬에 저장
            distance_matrix.at[sites_df['Station_Na'].iloc[i], sites_df['Station_Na'].iloc[j]] = distance

    return distance_matrix

def bfs_distance(graph, node_graph, start, end):
    # Create a queue for BFS and initialize it with the start edge and distance
    queue = deque([(start, 0)])
    # Create a set to keep track of visited edges
    visited = set()

    while queue:
        edge, distance = queue.popleft()

        if edge == end:
            # If the end edge is reached, return the distance
            return distance

        visited.add(edge)

        # Explore the neighbors of the current edge in the graph
        for neighbor, edge_length in graph[edge]:
            if np.isnan(edge_length):
                continue
            if neighbor not in visited:
                queue.append((neighbor, distance + edge_length))
                visited.add(neighbor)

        # Explore the neighbors of the current edge in the node graph
        for neighbor, edge_length in node_graph[edge]:
            if np.isnan(edge_length):
                continue
            if neighbor not in visited:
                queue.append((neighbor, distance + edge_length))
                visited.add(neighbor)

    # If no path is found between start and end edges, return infinity
    return float('inf')

In [17]:
# Compute the distance matrix
distance_matrix = compute_distance_matrix(sites_df, split_edges_df, relationships_df, noderelationships_df)

# Print the distance matrix
distance_matrix

Station_Na,평림댐,담양군(담양댐),담양군(금월교),담양군(덕용교),담양군(삼지교),담양군(양지교),담양군(광주댐),광주광역시(용산교),광주광역시(첨단대교),광주광역시(유촌교),광주광역시(풍영정천2교),광주광역시(어등대교),광주광역시(설월교),광주광역시(천교),광주광역시(극락교),장성군(용동교),장성군(장성댐),장성군(금계리),장성군(제2황룡교),광주광역시(용진교),장성군(수양저수지),광주광역시(평림교),광주광역시(장록교),담양군(중방4교),담양군(장천교),장성군(죽탄교)
Station_Na,,,,,,,,,,,,,,,,,,,,,,,,,,
평림댐,0,inf,inf,inf,inf,inf,inf,inf,inf,inf,inf,inf,inf,inf,inf,inf,inf,inf,inf,inf,inf,13.658982,22.684252,inf,inf,10.134144
담양군(담양댐),inf,0,12.244785,inf,12.766026,inf,inf,18.441292,28.77356,inf,inf,21.703338,inf,inf,23.075634,inf,inf,inf,inf,inf,inf,inf,inf,inf,inf,inf
담양군(금월교),inf,inf,0,inf,13.102279,inf,inf,18.777544,29.109813,inf,inf,22.039591,inf,inf,23.411887,inf,inf,inf,inf,inf,inf,inf,inf,inf,inf,inf
담양군(덕용교),inf,inf,inf,0,11.629695,inf,inf,17.304961,27.637229,inf,inf,20.567007,inf,inf,21.939303,inf,inf,inf,inf,inf,inf,inf,inf,inf,inf,inf
담양군(삼지교),inf,inf,inf,inf,0,inf,inf,7.455395,17.787663,inf,inf,10.717441,inf,inf,12.089738,inf,inf,inf,inf,inf,inf,inf,inf,inf,inf,inf
담양군(양지교),inf,inf,inf,inf,inf,0,inf,7.881201,18.213469,inf,inf,11.143247,inf,inf,12.515543,inf,inf,inf,inf,inf,inf,inf,inf,inf,inf,inf
담양군(광주댐),inf,inf,inf,inf,inf,8.761047,0,15.401337,25.733605,inf,inf,18.663383,inf,inf,20.03568,inf,inf,inf,inf,inf,inf,inf,inf,inf,inf,inf
광주광역시(용산교),inf,inf,inf,inf,inf,inf,inf,0,13.183655,inf,inf,6.113433,inf,inf,7.48573,inf,inf,inf,inf,inf,inf,inf,inf,inf,inf,inf
광주광역시(첨단대교),inf,inf,inf,inf,inf,inf,inf,inf,0,inf,inf,10.092622,inf,inf,11.464919,inf,inf,inf,inf,inf,inf,inf,inf,inf,inf,inf


In [18]:
# distance_matrix.to_csv("C:\\data_gnn_ver2\\adjacency.csv", encoding="utf-8-sig")

In [19]:
# Convert all entries in the DataFrame to float, replacing errors with np.inf
try:
    distance_matrix = distance_matrix.astype(float)
except ValueError:
    # If conversion fails, manually convert each value, replacing non-convertible values with np.inf
    distance_matrix = distance_matrix.applymap(lambda x: np.float(x) if pd.to_numeric(x, errors='coerce').notna().all() else np.inf)

# Now replace np.nan (and potentially missed non-numeric values) with np.inf
distance_matrix = distance_matrix.fillna(np.inf).replace([np.inf, -np.inf], 0)
#distance_matrix = distance_matrix.fillna(np.inf).replace([np.inf, -np.inf], np.inf)

# Convert the cleaned distance matrix to a NumPy array
distance_array = distance_matrix.to_numpy()

# Ensure the entire array is of a numeric type (float)
distance_array = np.where(np.isfinite(distance_array), distance_array, np.inf).astype(float)

# Create a figure and axis object for plotting
fig, ax = plt.subplots(figsize=(10, 10))

# Plot the distance matrix as a heatmap
sns.heatmap(distance_array, annot=True, cmap="YlGnBu", ax=ax, fmt='.1f')

# Set title and labels
ax.set_title("Distance Matrix", fontsize=16)
ax.set_xlabel("Sites", fontsize=14)
ax.set_ylabel("Sites", fontsize=14)

# Rotate x-axis labels for readability
plt.xticks(rotation=90)

# Display the plot
plt.show()

In [20]:
def transform_distance_matrix(matrix):
    # Create a copy of the matrix to avoid modifying the original
    transformed_matrix = matrix.copy()
    
    # Replace non-zero values with 5
    #transformed_matrix[transformed_matrix != 0] = 5
    
    # Make the matrix symmetric
    transformed_matrix = transformed_matrix + transformed_matrix.T
    
    return transformed_matrix

transformed_matrix = transform_distance_matrix(distance_array)
transformed_matrix

array([[ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        , 13.65898242, 22.68425173,  0.        ,  0.        ,
        20.2682873 ],
       [ 0.        ,  0.        , 12.24478534,  0.        , 12.76602594,
         0.        ,  0.        , 18.44129178, 28.7735601 ,  0.        ,
         0.        , 21.70333804,  0.        ,  0.        , 23.07563434,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ],
       [ 0.        , 12.24478534,  0.        ,  0.        , 13.10227865,
         0.        ,  0.        , 18.77754449, 29.10981281,  0.        ,
         0.        , 22.03959075,  0.        ,  0.        , 23.41188706,
       

In [21]:
# Create a figure and axis object for plotting
fig, ax = plt.subplots(figsize=(10, 10))

# Plot the distance matrix as a heatmap
sns.heatmap(transformed_matrix, annot=True, cmap="YlGnBu", ax=ax, fmt='.1f')

# Set title and labels
ax.set_title("Distance Matrix", fontsize=16)
ax.set_xlabel("Sites", fontsize=14)
ax.set_ylabel("Sites", fontsize=14)

# Rotate x-axis labels for readability
plt.xticks(rotation=90)

# Display the plot
plt.show()

In [22]:
#sites_df.to_csv("C:\data_gnn_ver2\sites_df.csv", encoding="utf-8-sig", index=False)